In [ ]:
from vetting_lib import *
from antares_client.search import get_thumbnails
import ipywidgets as widgets
from IPython.display import display, HTML
from io import BytesIO
import pandas as pd

In [ ]:
TAG = "lantern_xgboost_t2.0.7_c0.95"
CSV_PATH = "vetting_results.csv"
LIMIT = 100

print(f"Searching ANTARES for tag: {TAG} (limit {LIMIT}, skipping stellar)...")
loci = search_by_tag(TAG, limit=LIMIT, skip_stellar=True)
print(f"Found {len(loci)} non-stellar loci")

# Skip already-classified loci
done = load_classified_ids(CSV_PATH)
todo = [l for l in loci if l.locus_id not in done]
print(f"{len(todo)} unclassified")
for l in todo:
    print(f"  {l.locus_id}")

In [ ]:
import matplotlib.pyplot as plt

REASONS = ["", "saturation", "double star", "single AGN",
           "AGN + star", "cosmic rays", "bad pixels",
           "too red", "blue possible AGN", "diffraction spikes", "other"]

def show_locus(locus):
    """Display all vetting info for one locus."""
    _locus_data.clear()
    props = locus.properties or {}
    print(f'Locus ID: {locus.locus_id}')
    print(f'RA:       {locus.ra:.4f}')
    print(f'DEC:      {locus.dec:.4f}')
    print(f'Alerts:   {len(locus.alerts or [])}')
    # Lantern score
    max_score = props.get("lantern_xgboost_t2.0.7_c0.95_max_score")
    n_tagged = props.get("lantern_xgboost_t2.0.7_c0.95_num_tagged_alerts")
    print(f'Lantern max_score:  {max_score}')
    print(f'Lantern num_tagged: {n_tagged}')


    antares_url = f'https://antares.noirlab.edu/loci/{locus.locus_id}'
    legacy_url = f'https://www.legacysurvey.org/viewer?ra={locus.ra:.6f}&dec={locus.dec:.6f}&layer=ls-dr10&zoom=16'
    display(HTML(
        f'<a href="{antares_url}" target="_blank">ANTARES page</a> | '
        f'<a href="{legacy_url}" target="_blank">Legacy Survey viewer</a>'
    ))

    # Latest alert info (LSST only)
    alerts = locus.alerts or []
    lsst_alerts = [a for a in alerts if any(k.startswith('lsst_') for k in (a.properties or {}))]
    if lsst_alerts:
        alerts = lsst_alerts
    if alerts:
        aprops = alerts[-1].properties or {}
        band = aprops.get("ant_passband") or aprops.get("passband") or "?"

        brightest = props.get("brightest_alert_magnitude")
        newest = props.get("newest_alert_magnitude")
        print(f'\nDiff mag (brightest): {brightest:.2f} ({band})' if brightest else '\nDiff mag (brightest): -')
        print(f'Diff mag (newest):   {newest:.2f} ({band})' if newest else 'Diff mag (newest):   -')

        sci_flux = aprops.get("lsst_diaSource_scienceFlux")
        tmpl_flux = aprops.get("lsst_diaSource_templateFlux")
        diff_flux = aprops.get("lsst_diaSource_psfFlux")
        sci_mag = nJy_to_AB(sci_flux)
        tmpl_mag = nJy_to_AB(tmpl_flux)
        diff_mag = nJy_to_AB(diff_flux)
        if sci_mag:
            _locus_data['sci_mag'] = f'{sci_mag:.2f}'
        print(f'Science mag (latest):  {sci_mag:.2f} ({band})  ({sci_flux:.2e} nJy)' if sci_mag else 'Science mag (latest):  -')
        if tmpl_mag:
            _locus_data['tmpl_mag'] = f'{tmpl_mag:.2f}'
        print(f'Template mag (latest): {tmpl_mag:.2f} ({band})  ({tmpl_flux:.2e} nJy)' if tmpl_mag else 'Template mag (latest): -')
        print(f'Diff mag (latest):     {diff_mag:.2f} ({band})  ({diff_flux:.2e} nJy)' if diff_mag else 'Diff mag (latest):     -')

    # Catalog checks
    cats = list(locus.catalogs or [])
    print(f'\nCatalogs ({len(cats)}): {cats}')
    for c in ["milliquas", "gaia_dr3_variability", "gaia_dr3_gaia_source", "bright_guide_star_cat"]:
        print(f'  {c}: {"YES" if c in cats else "no"}')

    if "gaia_dr3_variability" in cats:
        for i, row in enumerate(locus.catalog_objects.get("gaia_dr3_variability", [])):
            print(f'  Variability {i}: class={row.get("class", "?")}, classifier={row.get("classifier", "?")}')

    if "milliquas" in cats:
        for i, row in enumerate(locus.catalog_objects.get("milliquas", [])):
            z = row.get("z")
            rmag = row.get("rmag")
            print(f'  Milliquas {i}: {row.get("name", "?")}, type={row.get("type", "?")}, '
                  f'z={z:.3f}, r={rmag:.2f}' if z and rmag else f'z={z}, r={rmag}')

    if "gaia_dr3_gaia_source" in cats:
        for i, row in enumerate(locus.catalog_objects.get("gaia_dr3_gaia_source", [])):
            g_mag = row.get("phot_g_mean_mag")
            bp_rp = row.get("bp_rp")
            print(f'  Gaia source {i}:')
            print(f'    G mag    = {g_mag:.2f}' if g_mag else '    G mag    = -')
            print(f'    BP-RP    = {bp_rp:.2f}' if bp_rp else '    BP-RP    = -')
            if bp_rp is not None:
                _locus_data['gaia_bp_rp'] = f'{bp_rp:.2f}'
            print(f'    pmra     = {fmt_sigma(row.get("pmra"), row.get("pmra_error"))} mas/yr')
            print(f'    pmdec    = {fmt_sigma(row.get("pmdec"), row.get("pmdec_error"))} mas/yr')
            print(f'    parallax = {fmt_sigma(row.get("parallax"), row.get("parallax_error"))} mas')

    # External queries
    print('\nQuerying PS1 DR2...')
    try:
        ps1_g, ps1_r, ps1_gr = ps1_gr_color(locus.ra, locus.dec)
        if ps1_gr is not None:
            print(f'  PS1 g={ps1_g:.2f}, r={ps1_r:.2f}, g-r={ps1_gr:.2f}')
            _locus_data['ps1_gr'] = f'{ps1_gr:.2f}'
        else:
            print('  No PS1 match (or missing g/r).')
    except Exception as e:
        print(f'  PS1 query error: {e}')

    print('Querying SkyMapper DR4...')
    try:
        sm_g, sm_r, sm_v, sm_gr, sm_vg = skymapper_colors(locus.ra, locus.dec)
        parts = []
        if sm_g: parts.append(f'g={sm_g:.2f}')
        if sm_r: parts.append(f'r={sm_r:.2f}')
        if sm_v: parts.append(f'v={sm_v:.2f}')
        if sm_gr is not None: parts.append(f'g-r={sm_gr:.2f}')
        if sm_vg is not None: parts.append(f'v-g={sm_vg:.2f}')
        print(f'  SkyMapper {", ".join(parts)}' if parts else '  No SkyMapper match.')
    except Exception as e:
        print(f'  SkyMapper error: {e}')

    print('Checking HST/JWST coverage...')
    coverage = {}
    try:
        coverage = hst_jwst_coverage(locus.ra, locus.dec)
        if coverage:
            for coll, info in coverage.items():
                print(f'  {coll}: {info["n_obs"]} obs, filters: {", ".join(info["filters"])}')
        else:
            print('  No HST/JWST imaging.')
    except Exception as e:
        print(f'  MAST error: {e}')


    # Light curve
    try:
        lc = locus.lightcurve
        if lc is not None and len(lc) > 0:
            tcol = next((c for c in ["ant_mjd", "mjd"] if c in lc.columns), None)
            mcol = next((c for c in ["ant_mag", "mag"] if c in lc.columns), None)
            if tcol and mcol:
                fig, ax = plt.subplots(figsize=(5, 2))
                ax.scatter(lc[tcol], lc[mcol], s=8)
                ax.invert_yaxis()
                ax.set_xlabel("MJD")
                ax.set_ylabel("mag")
                ax.set_title("Light curve")
                plt.tight_layout()
                plt.show()
    except Exception as e:
        print(f'  Light curve error: {e}')

    # Cutout images
    img_widgets = []

    if alerts:
        alert_id = alerts[-1].alert_id
        print(f'\nFetching thumbnails for alert: {alert_id}')
        thumbs = get_thumbnails(alert_id) or {}
        for ttype, t in thumbs.items():
            if "diff" in str(ttype).lower():
                lsst_blob = add_scale_bar(t["blob"], 0.2, fmt='png')
                img_widgets.append(widgets.VBox([
                    widgets.Label('LSST Diff (6")'),
                    widgets.Image(value=lsst_blob, format='png', width=200, height=200)
                ]))
                break
        else:
            print(f'  No difference thumbnail. Types: {list(thumbs.keys())}')

    print('Fetching DECaLS cutout...')
    try:
        decals_blob = get_decals_jpg(locus.ra, locus.dec)
        decals_blob = add_scale_bar(decals_blob, 0.262)
        img_widgets.append(widgets.VBox([
            widgets.Label('DECaLS DR10 grz (26")'),
            widgets.Image(value=decals_blob, format='jpeg', width=200, height=200)
        ]))
    except Exception as e:
        print(f'  DECaLS error: {e}')

    print('Fetching PS1 cutout...')
    try:
        ps1_blob, ps1_label = get_ps1_color_cutout(locus.ra, locus.dec)
        if ps1_blob:
            ps1_blob = add_scale_bar(ps1_blob, 120 * 0.25 / 256)
            img_widgets.append(widgets.VBox([
                widgets.Label(f'PS1 ({ps1_label}, 30")'),
                widgets.Image(value=ps1_blob, format='jpeg', width=200, height=200)
            ]))
        else:
            print('  No PS1 coverage.')
    except Exception as e:
        print(f'  PS1 cutout error: {e}')

    if coverage and 'HST' in coverage:
        print('Fetching HST cutout...')
        try:
            hst_blob = get_hst_cutout(locus.ra, locus.dec)
            if hst_blob:
                hst_blob = add_scale_bar(hst_blob, 0.003 * 3600 / 256)
                img_widgets.append(widgets.VBox([
                    widgets.Label('HST (11")'),
                    widgets.Image(value=hst_blob, format='jpeg', width=200, height=200)
                ]))
            else:
                print('  HST cutout not available via HiPS.')
        except Exception as e:
            print(f'  HST cutout error: {e}')

    if coverage and 'JWST' in coverage:
        print('Fetching JWST cutout...')
        try:
            jwst_blob = get_jwst_cutout(locus.ra, locus.dec)
            if jwst_blob:
                jwst_blob = add_scale_bar(jwst_blob, 0.003 * 3600 / 256)
                img_widgets.append(widgets.VBox([
                    widgets.Label('JWST (11")'),
                    widgets.Image(value=jwst_blob, format='jpeg', width=200, height=200)
                ]))
            else:
                print('  JWST cutout not available via HiPS.')
        except Exception as e:
            print(f'  JWST cutout error: {e}')

    print('Fetching Euclid cutout...')
    try:
        euclid_blob, euclid_label = get_euclid_cutout(locus.ra, locus.dec)
        if euclid_blob:
            from PIL import Image as _PILImage
            _eimg = _PILImage.open(BytesIO(euclid_blob))
            euclid_pxsc = 2 * 0.1 * 60 / _eimg.width
            euclid_blob = add_scale_bar(euclid_blob, euclid_pxsc, fmt='png')
            img_widgets.append(widgets.VBox([
                widgets.Label(f'Euclid ({euclid_label}, 12")'),
                widgets.Image(value=euclid_blob, format='png', width=200, height=200)
            ]))
        else:
            print('  No Euclid coverage.')
    except Exception as e:
        print(f'  Euclid error: {e}')

    if img_widgets:
        display(widgets.HBox(img_widgets))


# --- UI ---
_state = {'index': 0}
_locus_data = {}
output = widgets.Output()
counter_label = widgets.Label(value='')
reason_dropdown = widgets.Dropdown(options=REASONS, description='Reason:',
                                   layout=widgets.Layout(width='250px'))
notes_input = widgets.Textarea(placeholder='notes...',
                               layout=widgets.Layout(width='300px', height='40px'))

prev_btn = widgets.Button(description='Prev', layout=widgets.Layout(width='60px'))
next_btn = widgets.Button(description='Next', layout=widgets.Layout(width='60px'))
good_btn = widgets.Button(description='Good', button_style='success',
                          layout=widgets.Layout(width='80px'))
bad_btn = widgets.Button(description='Bad', button_style='danger',
                         layout=widgets.Layout(width='80px'))
skip_btn = widgets.Button(description='Skip', layout=widgets.Layout(width='80px'))

def _render():
    idx = _state['index']
    output.clear_output()
    reason_dropdown.value = ''
    notes_input.value = ''
    if not todo or idx >= len(todo):
        counter_label.value = f'done ({len(todo)})'
        with output:
            print('All loci classified.')
        return
    idx = max(0, idx)
    _state['index'] = idx
    counter_label.value = f'{idx + 1} / {len(todo)}'
    with output:
        show_locus(todo[idx])

def _save_grade(grade):
    if not todo:
        return
    locus = todo[_state['index']]
    save_to_csv(CSV_PATH, locus.locus_id, locus.ra, locus.dec,
                grade, reason_dropdown.value, notes_input.value,
                _locus_data.get('ps1_gr', ''),
                _locus_data.get('gaia_bp_rp', ''),
                _locus_data.get('sci_mag', ''),
                _locus_data.get('tmpl_mag', ''))
    print(f'Saved: {locus.locus_id} -> {grade}')
    _state['index'] += 1
    _render()

good_btn.on_click(lambda b: _save_grade('good'))
bad_btn.on_click(lambda b: _save_grade('bad'))
skip_btn.on_click(lambda b: (_state.update({'index': _state['index'] + 1}), _render()))
prev_btn.on_click(lambda b: (_state.update({'index': _state['index'] - 1}), _render()))
next_btn.on_click(lambda b: (_state.update({'index': _state['index'] + 1}), _render()))

display(widgets.HBox([prev_btn, next_btn, good_btn, bad_btn, skip_btn, counter_label]),
        reason_dropdown, notes_input, output)
_render()

In [ ]:
import os
if os.path.exists(CSV_PATH):
    display(pd.read_csv(CSV_PATH))
else:
    print("No classifications yet.")